# Week 02B — Camera Calibration in Practice

**MCTA 4364 Machine Vision** · Session 2 of Week 2 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W02_optics_calibration/W02B_camera_calibration.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W02_optics_calibration/W02B_camera_calibration.ipynb)

In Session A you wrote down the camera model $\lambda\mathbf{u} = K[R\,|\,\mathbf{t}]\mathbf{X}$ plus lens distortion. **Calibration** measures
the unknown numbers in that model ($f_x, f_y, c_x, c_y, k_1, k_2, p_1, p_2, k_3$) from images of a target with known geometry.
Once calibrated, a camera becomes a **measuring instrument**: you can remove distortion, find an object's pose and measure in
real units. We calibrate a real camera from the 13 chessboard photos that ship with OpenCV.

### Learning outcomes
By the end of this session you will be able to:
1. Explain **Zhang's method**: why several views of a planar target determine $K$ and distortion.
2. Detect chessboard corners with **sub-pixel** accuracy and run `cv2.calibrateCamera`.
3. Judge calibration quality with **reprojection error**, residual plots and view coverage.
4. **Undistort** images and verify the result quantitatively.
5. Estimate a target's **pose** (`solvePnP`), overlay 3D graphics, and **measure distances** on a plane.
6. Use **ArUco/ChArUco** markers for robust detection and pose estimation.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1 | Why and how we calibrate (Zhang's method) |
| 10–30 | 2–3 | Corner detection, calibration, error analysis, 3D view of the poses |
| 30–45 | 4–5 | Undistortion, straightness test; how many views? |
| 45–65 | 6–7 | Pose estimation, AR overlay, measuring on a plane |
| 65–72 | 8 | ArUco and ChArUco markers |
| 72–80 | 11 | Quiz and exit ticket |
| Home | 9–10 | Exercises with self-checks, calibrate your own webcam |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

from cvhelpers import SMOKE, check, data_dir, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
PATTERN = (9, 6)                        # inner corners per row, per column
NAMES = [f"left{i:02d}.jpg" for i in list(range(1, 10)) + list(range(11, 15))]
images = [load_sample(n) for n in NAMES]
gray_images = [cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) for im in images]
IMG_SIZE = gray_images[0].shape[::-1]   # (width, height)
print(f"{len(images)} calibration images of size {IMG_SIZE[0]} x {IMG_SIZE[1]}")

## 1. Why calibrate, and how Zhang's method works

Calibration is needed whenever pixels must be turned into **geometry**: measuring parts in millimetres, guiding a robot, stitching
panoramas, stereo 3D (Week 12) or estimating speed (Week 10).

**Zhang's method** (2000), used by OpenCV and MATLAB, needs only a flat printed pattern:
1. Define the target's corners in its own coordinate frame, all with $Z = 0$: $\mathbf{X}_j = (X_j, Y_j, 0)$.
2. For a plane, projection reduces to a **homography** $H = K[\mathbf{r}_1\ \mathbf{r}_2\ \mathbf{t}]$ (3×3), estimated from the corner matches in each image.
3. Because $\mathbf{r}_1, \mathbf{r}_2$ are orthonormal, each homography gives **two equations** on the 5 intrinsic unknowns. With ≥ 3 views
   (in general orientations), $K$ has a closed-form solution.
4. Finally all parameters, including distortion and every view's pose, are refined together by **non-linear least squares**
   (Levenberg–Marquardt), minimising the **reprojection error**:

$$ E = \sum_{\text{views } i}\ \sum_{\text{corners } j} \big\lVert \mathbf{u}_{ij} - \pi(K, \mathbf{d}, R_i, \mathbf{t}_i, \mathbf{X}_j) \big\rVert^2 $$

where $\pi$ is the full projection with distortion $\mathbf{d}$. OpenCV reports $\sqrt{E / N}$ (RMS error in pixels).
**Good practice:** 10–20 views, board tilted up to ~45° in different directions, covering the **whole image** including the corners
(where distortion is largest), sharp and evenly lit.

## 2. Finding the corners

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)     # 1 unit = 1 chessboard square

obj_points, img_points, coarse_points = [], [], []
for g in gray_images:
    found, corners = cv2.findChessboardCorners(g, PATTERN)
    assert found, "chessboard not found"
    corners = corners.reshape(-1, 1, 2)      # OpenCV 4.x returns (N, 1, 2), OpenCV 5.x returns (N, 2)
    coarse_points.append(corners.copy())
    refined = cv2.cornerSubPix(g, corners, (11, 11), (-1, -1), criteria)
    obj_points.append(objp)
    img_points.append(refined)

fig, axes = plt.subplots(3, 5, figsize=(20, 9.5))
for ax, im, pts, name in zip(axes.ravel(), images, img_points, NAMES):
    vis = im.copy()
    cv2.drawChessboardCorners(vis, PATTERN, pts, True)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis("off")
cover = np.zeros(IMG_SIZE[::-1], np.float32)
for pts in img_points:
    hull = cv2.convexHull(pts.astype(np.int32))
    layer = np.zeros_like(cover); cv2.fillConvexPoly(layer, hull, 1); cover += layer
axes[2, 3].imshow(cover, cmap="viridis"); axes[2, 3].set_title("coverage: how many views\ncover each pixel")
axes[2, 3].axis("off")
axes[2, 4].axis("off")
plt.suptitle("Figure 2.1 - Detected corners in all 13 views, and image coverage", fontweight="bold")
plt.tight_layout(); plt.show()

**Sub-pixel refinement.** `findChessboardCorners` returns corners to about ±0.5 px. `cornerSubPix` refines them by exploiting the fact that,
at the true corner $\mathbf{q}$, the image gradient at every nearby point $\mathbf{p}$ is perpendicular to $\mathbf{p} - \mathbf{q}$. Solving this small least-squares
problem gives accuracies of ~0.05 px, which matters because every calibrated number inherits the corner accuracy.

In [ ]:
view, k = 0, 20
c0, c1 = coarse_points[view][k, 0], img_points[view][k, 0]
x0, y0 = int(c1[0]) - 8, int(c1[1]) - 8
patch = gray_images[view][y0:y0 + 17, x0:x0 + 17]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(patch, cmap="gray", extent=(x0 - 0.5, x0 + 16.5, y0 + 16.5, y0 - 0.5))
ax.plot(*c0, "o", ms=12, mfc="none", mec="#fbbc04", mew=2.5, label=f"coarse ({c0[0]:.2f}, {c0[1]:.2f})")
ax.plot(*c1, "+", ms=20, color="#d93025", mew=2.5, label=f"sub-pixel ({c1[0]:.2f}, {c1[1]:.2f})")
ax.legend(loc="lower left", fontsize=8); ax.set_title("Figure 2.2 - One corner, 17 x 17 pixel zoom")
plt.show()
shift = np.linalg.norm(np.vstack([c - r for c, r in zip(coarse_points, img_points)]).reshape(-1, 2), axis=1)
print(f"Sub-pixel refinement moved corners by {shift.mean():.2f} px on average (max {shift.max():.2f} px)")

## 3. Calibrate and evaluate

In [ ]:
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points, IMG_SIZE, None, None)
np.set_printoptions(precision=3, suppress=True)
print(f"RMS reprojection error: {rms:.3f} px\n")
print("K =\n", K)
print("\ndistortion (k1, k2, p1, p2, k3) =", dist.ravel())
fovx, fovy, *_ = cv2.calibrationMatrixValues(K, IMG_SIZE, 1.0, 1.0)
print(f"\nField of view: {fovx:.1f} x {fovy:.1f} degrees | principal point offset from centre: "
      f"({K[0, 2] - IMG_SIZE[0] / 2:.1f}, {K[1, 2] - IMG_SIZE[1] / 2:.1f}) px")

In [ ]:
per_view, residuals = [], []
for op, ip, rv, tv in zip(obj_points, img_points, rvecs, tvecs):
    proj, _ = cv2.projectPoints(op, rv, tv, K, dist)
    d = (ip - proj).reshape(-1, 2)
    residuals.append(d)
    per_view.append(np.sqrt((d ** 2).sum(1).mean()))
per_view = np.array(per_view)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8), gridspec_kw={"width_ratios": [1.3, 1, 1]})
axes[0].bar(range(len(NAMES)), per_view, color=["#d93025" if e > 1.5 * np.median(per_view) else "#1a73e8" for e in per_view])
axes[0].axhline(rms, color="k", ls="--", label=f"overall RMS {rms:.3f} px"); axes[0].legend()
axes[0].set_xticks(range(len(NAMES)), [n[:-4] for n in NAMES], rotation=60); axes[0].set_ylabel("RMS error (px)")
axes[0].set_title("per-view reprojection error (red = suspicious view)")
worst = int(per_view.argmax())
pts = img_points[worst].reshape(-1, 2); d = residuals[worst]
axes[1].imshow(images[worst][..., ::-1]); axes[1].quiver(pts[:, 0], pts[:, 1], d[:, 0], d[:, 1], color="#fbbc04",
                                                        angles="xy", scale_units="xy", scale=0.02)
axes[1].set_title(f"residuals of worst view {NAMES[worst]} (x50)"); axes[1].axis("off")
allres = np.vstack(residuals)
axes[2].scatter(allres[:, 0], allres[:, 1], s=4, alpha=0.5); axes[2].add_patch(plt.Circle((0, 0), rms, fill=False, color="#d93025"))
axes[2].set_aspect("equal"); axes[2].grid(alpha=0.3); axes[2].set_xlabel("x error (px)"); axes[2].set_ylabel("y error (px)")
axes[2].set_title("all residuals: should be small, round and centred")
plt.suptitle("Figure 3.1 - Calibration quality", fontweight="bold")
plt.tight_layout(); plt.show()

**Practical step: inspect and remove bad views.** Views flagged in red have errors well above the median. Looking at the residual plot
of the worst view reveals a few corners with errors of several pixels. `cornerSubPix` jumped to a wrong position where the board edge is
close to the corner. A single bad view pulls every parameter, so good practice is to remove it (or fix the detection) and recalibrate.

In [ ]:
keep = [i for i, e in enumerate(per_view) if e <= 1.5 * np.median(per_view)]
rms_clean, K_clean, dist_clean, _, _ = cv2.calibrateCamera([obj_points[i] for i in keep], [img_points[i] for i in keep],
                                                           IMG_SIZE, None, None)
print(f"removed views: {[NAMES[i] for i in range(len(NAMES)) if i not in keep]}")
print(f"RMS error {rms:.3f} px -> {rms_clean:.3f} px")
print(f"fx {K[0, 0]:.1f} -> {K_clean[0, 0]:.1f} px | k1 {dist.ravel()[0]:.3f} -> {dist_clean.ravel()[0]:.3f}")
print("We keep the full calibration below so that everyone gets the same numbers, but in a real project you would use the cleaned one.")

**Figure 3.2 — Where were the boards?** Each coloured quadrilateral is one calibration view, drawn in the camera coordinate frame (units: chessboard squares). Rotate the plot: good calibration sets show many different distances and tilts.

In [ ]:
fig = go.Figure()
board_outline = np.array([[0, 0, 0], [8, 0, 0], [8, 5, 0], [0, 5, 0], [0, 0, 0]], float)
palette = ["#1a73e8", "#d93025", "#188038", "#fbbc04", "#a142f4", "#e37400", "#12b5cb"]
for i, (rv, tv) in enumerate(zip(rvecs, tvecs)):
    R, _ = cv2.Rodrigues(rv)
    P = (R @ board_outline.T).T + tv.ravel()
    fig.add_trace(go.Scatter3d(x=P[:, 0], y=P[:, 2], z=-P[:, 1], mode="lines", line=dict(color=palette[i % 7], width=5),
                               name=NAMES[i][:-4]))
s = 6.0
fr = np.array([[(u - K[0, 2]) / K[0, 0] * s, (v - K[1, 2]) / K[1, 1] * s, s] for u, v in [(0, 0), (640, 0), (640, 480), (0, 480)]])
for c in fr:
    fig.add_trace(go.Scatter3d(x=[0, c[0]], y=[0, c[2]], z=[0, -c[1]], mode="lines", line=dict(color="black"), showlegend=False))
fig.update_layout(title="Figure 3.2 - Estimated board poses relative to the camera (interactive)", height=560,
                  scene=dict(xaxis_title="X", yaxis_title="Z (depth)", zaxis_title="-Y", aspectmode="data"))
fig.show()

## 4. Undistortion

With $K$ and the distortion coefficients we can remove the lens distortion. `cv2.getOptimalNewCameraMatrix` chooses the output camera:
**alpha = 0** crops so every output pixel is valid; **alpha = 1** keeps every source pixel (black borders appear). For video, compute the
remapping once with `cv2.initUndistortRectifyMap` and apply `cv2.remap` to every frame, which is much faster than calling `undistort` repeatedly.

A quantitative check: the corners of each chessboard row lie on a **straight line** in the world, so after undistortion they should also be
collinear in the image. We fit a line to each row and measure the RMS deviation.

In [ ]:
idx = NAMES.index("left12.jpg")
K0, _ = cv2.getOptimalNewCameraMatrix(K, dist, IMG_SIZE, 0)
K1, _ = cv2.getOptimalNewCameraMatrix(K, dist, IMG_SIZE, 1)
und0 = cv2.undistort(images[idx], K, dist, None, K0)
und1 = cv2.undistort(images[idx], K, dist, None, K1)
show(images[idx], und0, und1, titles=["original (distorted)", "undistorted, alpha = 0 (cropped)", "undistorted, alpha = 1 (all pixels)"],
     figsize=(20, 5))


def straightness(points):
    """RMS distance (px) of chessboard-row corners from their best-fit lines."""
    pts = points.reshape(PATTERN[1], PATTERN[0], 2)
    errs = []
    for row in pts:
        mean = row.mean(0)
        _, _, vt = np.linalg.svd(row - mean)
        normal = vt[1]
        errs.append((row - mean) @ normal)
    return float(np.sqrt(np.mean(np.square(errs))))


before = [straightness(p) for p in img_points]
after = [straightness(cv2.undistortPoints(p, K, dist, P=K)) for p in img_points]
x = np.arange(len(NAMES))
plt.figure(figsize=(12, 3.6))
plt.bar(x - 0.2, before, 0.4, label="original corners", color="#d93025")
plt.bar(x + 0.2, after, 0.4, label="after undistortion", color="#188038")
plt.xticks(x, [n[:-4] for n in NAMES], rotation=60); plt.ylabel("row straightness error (px)"); plt.legend()
plt.title(f"Figure 4.1 - Straight lines become straight: mean error {np.mean(before):.2f} -> {np.mean(after):.3f} px", fontweight="bold")
plt.show()

In [ ]:
map_x, map_y = cv2.initUndistortRectifyMap(K, dist, None, K, IMG_SIZE, cv2.CV_32FC1)
uu, vv = np.meshgrid(np.arange(IMG_SIZE[0]), np.arange(IMG_SIZE[1]))
shift_px = np.hypot(map_x - uu, map_y - vv)
plt.figure(figsize=(8, 5))
plt.imshow(shift_px, cmap="magma"); plt.colorbar(label="correction (pixels)")
plt.contour(shift_px, levels=[1, 2, 5, 10], colors="w", linewidths=0.8)
plt.title("Figure 4.2 - How far each pixel moves during undistortion", fontweight="bold"); plt.axis("off"); plt.show()
print(f"Largest correction: {shift_px.max():.1f} px (in the corners). Measurements near the image edge MUST be undistorted.")

## 5. How many views do you need?

We repeat the calibration many times using random subsets of *n* views and look at how much the estimated focal length varies.
Few views give unstable, unreliable estimates, even when their own reprojection error looks small.

In [ ]:
rng = np.random.default_rng(0)
trials = 4 if SMOKE else 15
ns = list(range(3, len(NAMES) + 1, 1 if not SMOKE else 3))
fx_stats, rms_stats = [], []
for n in ns:
    fx_vals, rms_vals = [], []
    for _ in range(trials):
        sel = rng.choice(len(NAMES), n, replace=False)
        r_, K_, *_ = cv2.calibrateCamera([obj_points[i] for i in sel], [img_points[i] for i in sel], IMG_SIZE, None, None)
        fx_vals.append(K_[0, 0]); rms_vals.append(r_)
    fx_stats.append((np.mean(fx_vals), np.std(fx_vals))); rms_stats.append(np.mean(rms_vals))
fx_stats = np.array(fx_stats)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].errorbar(ns, fx_stats[:, 0], yerr=fx_stats[:, 1], marker="o", capsize=4, lw=2)
axes[0].axhline(K[0, 0], color="k", ls="--", label="all 13 views"); axes[0].legend()
axes[0].set_xlabel("number of views"); axes[0].set_ylabel("estimated fx (px)"); axes[0].grid(alpha=0.3)
axes[0].set_title("focal-length estimate (mean +/- std over random subsets)")
axes[1].plot(ns, rms_stats, "s-", color="#e37400"); axes[1].set_xlabel("number of views"); axes[1].set_ylabel("RMS error (px)")
axes[1].grid(alpha=0.3); axes[1].set_title("training RMS error barely changes: it does NOT reveal the instability")
plt.suptitle("Figure 5.1 - More views, more reliable intrinsics", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Pose estimation and an AR overlay

Given a calibrated camera and the known 3D model of an object (here: the board), `cv2.solvePnP` finds the rotation and translation that
best explain the observed points: the **Perspective-n-Point** problem. With the pose we can project any 3D geometry onto the image:
the basis of augmented reality and robot guidance (pick the part at this pose).

In [ ]:
def draw_pose(img, rvec, tvec, Kc, dc):
    out = img.copy()
    cube = np.float32([[0, 0, 0], [3, 0, 0], [3, 3, 0], [0, 3, 0], [0, 0, -3], [3, 0, -3], [3, 3, -3], [0, 3, -3]])
    p, _ = cv2.projectPoints(cube, rvec, tvec, Kc, dc)
    p = p.reshape(-1, 2).astype(int)
    cv2.drawContours(out, [p[:4]], -1, (0, 200, 0), -3)
    for i, j in zip(range(4), range(4, 8)):
        cv2.line(out, tuple(p[i]), tuple(p[j]), (255, 0, 0), 2)
    cv2.drawContours(out, [p[4:]], -1, (0, 0, 255), 2)
    cv2.drawFrameAxes(out, Kc, dc, rvec, tvec, 2, 3)
    return out


vis = []
for i in (0, 5, 11):
    ok, rv, tv = cv2.solvePnP(objp, img_points[i], K, dist)
    vis.append(draw_pose(images[i], rv, tv, K, dist))
    R_board, _ = cv2.Rodrigues(rv)
    tilt = np.degrees(np.arccos(abs(R_board[2, 2])))     # angle between board normal and optical axis
    print(f"{NAMES[i]}: board origin at {tv.ravel().round(2)} squares from the camera, tilted {tilt:.0f} deg")
show(*vis, titles=[f"{NAMES[i]}: pose + virtual cube" for i in (0, 5, 11)], figsize=(20, 5))

## 7. Measuring on a plane

For points on a known plane (a conveyor belt, a table, the board), a single calibrated camera can measure real distances:
undistort the pixel coordinates, then map them to the plane with the **homography** $H$ from image to plane.

A realistic workflow: mark **four reference points** with known positions on the plane (here, the 4 outer corners of the board), fit $H$
from those four points only, and then measure anything else. We test it on all 93 neighbouring-corner distances, whose ground truth is exactly
**1 square**. (With a printed board of 25 mm squares, multiply by 25 mm.)

In [ ]:
view = NAMES.index("left12.jpg")
REF = [0, PATTERN[0] - 1, PATTERN[0] * (PATTERN[1] - 1), PATTERN[0] * PATTERN[1] - 1]     # the 4 outer corners
und_pts = cv2.undistortPoints(img_points[view], K, dist, P=K).reshape(-1, 2)
H_img2plane = cv2.getPerspectiveTransform(und_pts[REF].astype(np.float32), objp[REF, :2].astype(np.float32))


def to_plane(uv_distorted):
    und = cv2.undistortPoints(np.float32(uv_distorted).reshape(-1, 1, 2), K, dist, P=K)
    return cv2.perspectiveTransform(und, H_img2plane).reshape(-1, 2)


plane = to_plane(img_points[view])            # every corner, measured with H from only 4 reference points
grid = plane.reshape(PATTERN[1], PATTERN[0], 2)
d_h = np.linalg.norm(np.diff(grid, axis=1), axis=2).ravel()
d_v = np.linalg.norm(np.diff(grid, axis=0), axis=2).ravel()
raw = img_points[view].reshape(PATTERN[1], PATTERN[0], 2)
px_h = np.linalg.norm(np.diff(raw, axis=1), axis=2).ravel()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].hist(np.r_[d_h, d_v], bins=30, color="#188038")
axes[0].axvline(1.0, color="k", ls="--"); axes[0].set_xlabel("measured neighbour distance (squares)")
axes[0].set_title(f"calibrated measurement: {np.r_[d_h, d_v].mean():.4f} +/- {np.r_[d_h, d_v].std():.4f} squares")
axes[1].hist(px_h, bins=30, color="#d93025"); axes[1].set_xlabel("same distances in raw pixels")
axes[1].set_title(f"raw pixels vary {px_h.min():.1f}-{px_h.max():.1f} px because of perspective")
plt.suptitle(f"Figure 7.1 - Metric measurement on the board plane ({NAMES[view]})", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: measure between any two corners
Pick two corners (index 0–53). The calibrated measurement is compared with the true distance on the board.

In [ ]:
def measure_demo(corner_a=0, corner_b=53):
    pa, pb = img_points[view][corner_a, 0], img_points[view][corner_b, 0]
    wa, wb = to_plane([pa, pb])
    measured = np.linalg.norm(wa - wb)
    truth = np.linalg.norm(objp[corner_a, :2] - objp[corner_b, :2])
    vis = images[view].copy()
    cv2.line(vis, tuple(pa.astype(int)), tuple(pb.astype(int)), (0, 0, 255), 2)
    for p in (pa, pb):
        cv2.circle(vis, tuple(p.astype(int)), 6, (0, 255, 255), 2)
    show(vis, titles=[f"measured {measured:.3f} squares, true {truth:.3f} (error {abs(measured - truth) / truth * 100:.2f}%), "
                      f"pixels {np.linalg.norm(pa - pb):.1f}"], figsize=(9, 6.5))


interact(measure_demo, corner_a=widgets.IntSlider(value=0, min=0, max=53), corner_b=widgets.IntSlider(value=53, min=0, max=53))

## 8. ArUco and ChArUco markers

A chessboard must be **fully visible** to be detected, and its orientation is ambiguous. **Fiducial markers** solve both problems:
- **ArUco** / **AprilTag** markers are square binary codes with a unique ID. Each marker alone gives its 4 corners and therefore a pose.
  They are used for robot localisation, tool tracking and AR.
- A **ChArUco** board combines a chessboard (precise corners) with ArUco markers in the white squares (unique IDs), so it can be detected
  even when partly hidden or cropped. This is the recommended target for modern calibration.

Below we render a ChArUco board seen by *our calibrated camera* at a **known pose**, detect it, and check that `solvePnP` recovers that pose.

In [ ]:
aruco = cv2.aruco
dictionary = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
SQUARE, MARKER = 1.0, 0.7                                   # in "squares", like the chessboard above
board = aruco.CharucoBoard((7, 5), SQUARE, MARKER, dictionary)
board_px = 100                                              # pixels per square in the printable image
board_img = board.generateImage((7 * board_px, 5 * board_px), marginSize=0)

# Render the board with the calibrated camera at a known pose: plane point (X, Y, 0) -> image via H = K [r1 r2 t]
rvec_true = np.array([0.35, -0.45, 0.15]); tvec_true = np.array([-3.5, -2.5, 9.0])
R_true, _ = cv2.Rodrigues(rvec_true)
S = np.diag([1 / board_px, 1 / board_px, 1.0])              # board-image pixels -> squares
H_render = K @ np.column_stack([R_true[:, 0], R_true[:, 1], tvec_true]) @ S
rng = np.random.default_rng(3)
background = cv2.GaussianBlur(rng.integers(60, 200, (IMG_SIZE[1], IMG_SIZE[0])).astype(np.uint8), (0, 0), 3)
warped = cv2.warpPerspective(board_img, H_render, IMG_SIZE, flags=cv2.INTER_LINEAR)
mask = cv2.warpPerspective(np.full_like(board_img, 255), H_render, IMG_SIZE)
scene = np.where(mask > 0, warped, background).astype(np.uint8)
scene = np.clip(scene + rng.normal(0, 4, scene.shape), 0, 255).astype(np.uint8)
scene[:, 420:] = background[:, 420:]                        # occlude the right side of the board!

det = aruco.CharucoDetector(board)
ch_corners, ch_ids, m_corners, m_ids = det.detectBoard(scene)
vis = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
aruco.drawDetectedMarkers(vis, m_corners, m_ids)
if ch_corners is not None:
    ch_corners = ch_corners.reshape(-1, 1, 2)       # same (N, 1, 2) convention on OpenCV 4.x and 5.x
    aruco.drawDetectedCornersCharuco(vis, ch_corners, ch_ids, (0, 0, 255))
    obj, img = board.matchImagePoints(ch_corners, ch_ids)
    ok, rv, tv = cv2.solvePnP(obj, img, K, None)
    cv2.drawFrameAxes(vis, K, None, rv, tv, 2, 3)
    print(f"markers found: {0 if m_ids is None else len(m_ids)} | ChArUco corners: {len(ch_ids)} (board partly hidden)")
    print("true translation     :", tvec_true.round(3), "| rotation", rvec_true.round(3))
    print("estimated translation:", tv.ravel().round(3), "| rotation", rv.ravel().round(3))
show(board_img, vis, titles=["printable ChArUco board (7 x 5, DICT_4X4_50)", "rendered view, occluded, detected, pose recovered"],
     figsize=(17, 5.6))

**Printable targets.** Save `board_img` (or a chessboard) with `cv2.imwrite` and print it at 100% scale on stiff card. Measure the printed square size with a ruler: calibration accuracy depends on it.

## 9. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Reprojection error
Compute the overall RMS reprojection error (pixels) exactly as `cv2.calibrateCamera` reports it: the square root of the mean *squared Euclidean distance* over all corners of all views.

In [ ]:
def reprojection_rmse(obj_pts, img_pts, K, dist, rvecs, tvecs):
    # TODO: your code here (cv2.projectPoints is allowed)
    raise NotImplementedError

In [ ]:
def _test_e1():
    got = reprojection_rmse(obj_points, img_points, K, dist, rvecs, tvecs)
    assert abs(got - rms) < 1e-4, f"expected {rms:.5f} px (from calibrateCamera), got {got:.5f}"


check("E1 reprojection_rmse", _test_e1)

### E2. Apply a homography
Map `(N, 2)` pixel points through a 3×3 homography `H` (homogeneous coordinates, then divide by the third component). Do **not** use `cv2.perspectiveTransform`.

In [ ]:
def apply_homography(H, pts):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    pts = np.random.default_rng(4).uniform(0, 640, (40, 2))
    for Hm in (H_img2plane, np.linalg.inv(H_img2plane), H_render):
        got = np.asarray(apply_homography(Hm, pts))
        ref = cv2.perspectiveTransform(pts.reshape(-1, 1, 2), Hm).reshape(-1, 2)
        assert got.shape == (40, 2), f"expected shape (40, 2), got {got.shape}"
        assert np.allclose(got, ref, atol=1e-6), "results differ from cv2.perspectiveTransform: did you divide by w?"


check("E2 apply_homography", _test_e2)

### E3. Field of view from K
Return the horizontal and vertical field of view in **degrees** for an intrinsic matrix `K` and image size `(width, height)`.

In [ ]:
def fov_from_K(K, image_size):
    # TODO: return (fov_x_deg, fov_y_deg)
    raise NotImplementedError

In [ ]:
def _test_e3():
    for Kt, size in [(K, IMG_SIZE), (np.array([[1000, 0, 960], [0, 1000, 540], [0, 0, 1.0]]), (1920, 1080))]:
        fx_deg, fy_deg = fov_from_K(Kt, size)
        ref = cv2.calibrationMatrixValues(Kt, size, 1.0, 1.0)
        assert abs(fx_deg - ref[0]) < 1e-6 and abs(fy_deg - ref[1]) < 1e-6, \
            f"expected ({ref[0]:.3f}, {ref[1]:.3f}) deg, got ({fx_deg:.3f}, {fy_deg:.3f}). Hint: the principal point need not be centred."


check("E3 fov_from_K", _test_e3)

### E4. Bad calibration sets (no self-check)
Calibrate using only the **4 most frontal** views (smallest rotation angle). Compare K, the distortion coefficients and the straightness
test of Section 4 with the full calibration. What goes wrong, and why?

In [ ]:
# E4: your code here

## 10. Challenge: calibrate your own camera

1. Save a printable chessboard: `cv2.imwrite("chessboard.png", board)` with the code below, print it and glue it to card.
2. With `RUN_WEBCAM = True`, capture ≥ 15 views (move the board around the whole image, tilt it up to ~45°).
3. Calibrate, report RMS error, K, distortion and field of view. Plot the per-view errors and remove bad views.
4. Place a coin on the board and **measure its diameter in millimetres** using Section 7. Compare with a ruler.

In [ ]:
square_px = 80
chess = np.kron((np.indices((7, 10)).sum(0) % 2) * 255, np.ones((square_px, square_px))).astype(np.uint8)
chess = cv2.copyMakeBorder(chess, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255)
cv2.imwrite(str(data_dir() / "chessboard_9x6.png"), chess)
print("Printable 9 x 6 inner-corner chessboard saved to", data_dir() / "chessboard_9x6.png")

RUN_WEBCAM = False
if RUN_WEBCAM:
    cap, my_obj, my_img = cv2.VideoCapture(0), [], []
    while len(my_img) < 15:
        ok_cam, frame = cap.read()
        if not ok_cam:
            break
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(g, PATTERN)
        if found and input(f"view {len(my_img) + 1}: keep? [y/n] ") == "y":
            my_obj.append(objp); my_img.append(cv2.cornerSubPix(g, corners.reshape(-1, 1, 2), (11, 11), (-1, -1), criteria))
    cap.release()
    print(cv2.calibrateCamera(my_obj, my_img, g.shape[::-1], None, None)[:3])

## 11. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": "What is the minimum number of views of a planar target needed to solve for all 5 intrinsic parameters with Zhang's method (in general)?",
        "options": [
            '1',
            '2',
            '3',
            '10',
        ],
        "answer": 'b44cd1596a',
        "explain": "Each view's homography gives two constraints on the 5 unknowns of K.",
    },
    {
        "q": 'Your calibration reports an RMS reprojection error of 0.15 px using only 3 nearly frontal views. What should you conclude?',
        "options": [
            'The calibration is excellent',
            'The low error does not prove accuracy: add more, tilted views covering the whole image',
            'The lens has no distortion',
            'The camera is broken',
        ],
        "answer": '2a299415ef',
        "explain": 'Figure 5.1: few views can fit their own data well while the intrinsics are poorly constrained.',
    },
    {
        "q": 'Why is cornerSubPix used after findChessboardCorners?',
        "options": [
            'To find more corners',
            'To refine corner positions to a fraction of a pixel',
            'To remove distortion',
            'To speed up calibration',
        ],
        "answer": 'a31d6a00dd',
        "explain": 'Every calibrated parameter inherits the accuracy of the corners.',
    },
    {
        "q": 'Which parameters change when the camera is moved to a new position?',
        "options": [
            'K and distortion',
            'Only the extrinsics R and t',
            'All parameters',
            'None',
        ],
        "answer": '2afb05faf3',
        "explain": 'Intrinsics describe the camera and lens; they stay the same as long as focus and zoom are unchanged.',
    },
    {
        "q": 'What does cv2.solvePnP compute?',
        "options": [
            'The intrinsic matrix',
            'The pose of a known 3D object relative to the calibrated camera',
            'Distortion coefficients',
            'A homography between two images',
        ],
        "answer": '1cf240a972',
        "explain": 'Perspective-n-Point: 3D model points + their image points + K -> R, t.',
    },
    {
        "q": 'Why are ChArUco boards preferred over plain chessboards for calibration?',
        "options": [
            'They are cheaper to print',
            'Corners have unique IDs, so the board can be partly hidden or cropped',
            'They need no camera model',
            'They work without lighting',
        ],
        "answer": 'afd0c3f2f9',
        "explain": 'Plain chessboards must be completely visible and are orientation-ambiguous.',
    },
    {
        "q": 'A single calibrated camera can measure metric distances between two points when...',
        "options": [
            'always',
            'the points lie on a known plane (or their depth is known)',
            'the image is in colour',
            'the lens is telecentric only',
        ],
        "answer": '3cfc3cfcdd',
        "explain": "Otherwise depth is ambiguous along each pixel's ray: you need stereo or another constraint (Week 12).",
    },
], title='Week 02B quiz')

## 12. Exit ticket (reflection)
1. After calibrating, a technician refocuses the lens. Which parameters are now invalid, and what must be done?
2. Why must calibration views cover the image corners?
3. Name two sources of error in the Section 7 plane measurement that calibration does **not** remove.

## 13. What's new (2025–26)

> - **Learned calibration and camera-parameter estimation.** Networks such as GeoCalib (2024) estimate focal length and distortion from a single
>   ordinary photo, which is useful for internet images, although target-based calibration remains the standard for measurement.
> - **3D foundation models** (Week 12, e.g. DUSt3R/MASt3R and VGGT) jointly estimate cameras and 3D structure from a few uncalibrated images.
> - OpenCV now bundles **AprilTag** dictionaries in `cv2.aruco` (e.g. `DICT_APRILTAG_36h11`) alongside ArUco, and `findChessboardCornersSB` offers a more robust, more accurate corner detector.

## 14. References and further exploration

**Seminal papers**
- Zhang, Z. (2000). *A flexible new technique for camera calibration.* IEEE TPAMI 22(11). [doi:10.1109/34.888718](https://doi.org/10.1109/34.888718)
- Tsai, R. Y. (1987). *A versatile camera calibration technique for high-accuracy 3D machine vision metrology.* IEEE J. Robotics and Automation 3(4). [doi:10.1109/JRA.1987.1087109](https://doi.org/10.1109/JRA.1987.1087109)
- Garrido-Jurado, S. et al. (2014). *Automatic generation and detection of highly reliable fiducial markers under occlusion* (ArUco). Pattern Recognition 47(6). [doi:10.1016/j.patcog.2014.01.005](https://doi.org/10.1016/j.patcog.2014.01.005)
- Olson, E. (2011). *AprilTag: A robust and flexible visual fiducial system.* ICRA. [doi:10.1109/ICRA.2011.5979561](https://doi.org/10.1109/ICRA.2011.5979561)

**Textbooks**
- Hartley & Zisserman (2004). *Multiple View Geometry*, ch. 7 (computing the camera matrix) and ch. 8.
- Corke, P. (2023). *Robotics, Vision and Control*, 3rd ed., ch. 13 (camera modelling and calibration). Springer.
- Szeliski (2022), §11.1 (geometric intrinsic calibration). [szeliski.org/Book](https://szeliski.org/Book/)

**Documentation and tools**
- OpenCV tutorials: *Camera calibration*, *Pose estimation*, *Detection of ArUco and ChArUco*. [docs.opencv.org](https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html)
- calib.io: *Camera calibration pattern generator* (printable targets). [calib.io](https://calib.io/pages/camera-calibration-pattern-generator)
- Bouguet, J.-Y. *Camera Calibration Toolbox for MATLAB* (classic reference implementation). [data.caltech.edu](https://data.caltech.edu/records/jx9cx-fdh55)

**Images:** `left01.jpg`–`left14.jpg` (OpenCV sample data). The ChArUco scene is rendered in code.

## 15. Key takeaways
- Calibration estimates **K** and **distortion** from several views of a known target (Zhang's method), minimising **reprojection error**.
- Use **sub-pixel corners**, 10–20 **tilted** views covering the **whole image**; a low RMS alone does not prove a good calibration.
- **Undistort** before measuring; verify with a straightness test. Precompute `initUndistortRectifyMap` for video.
- `solvePnP` gives an object's **pose**; on a known plane a single camera can **measure in real units** via a homography.
- **ChArUco/ArUco/AprilTag** markers add unique IDs: robust to occlusion, ideal for calibration and robot localisation.

**Next week (03A):** point operations (contrast, gamma, histogram equalisation) and geometric transforms (affine, homography warping).